In [ ]:
# Cell 2: Upload CSV File (Run this cell and select your file)
from google.colab import files
import pandas as pd
import io

print("="*50)
print("UPLOAD YOUR STUDENT DATA CSV FILE")
print("="*50)
print("Please upload 'student_data.csv' when prompted...")

uploaded = files.upload()

# Read the uploaded file
for filename in uploaded.keys():
    print(f"\n✅ File '{filename}' uploaded successfully!")
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    print(f"📊 Dataset loaded with {len(df)} students")

print("\nFirst 5 rows of your data:")
df.head()

In [ ]:
# Cell 3: Import All Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ All libraries imported successfully!")
print("📚 Libraries: Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn")

In [ ]:
# Cell 4: Dataset Overview
print("="*50)
print("DATASET INFORMATION")
print("="*50)

print(f"\n📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📋 Column Names:")
for col in df.columns:
    print(f"   • {col}")

print(f"\n🔍 Data Types:")
print(df.dtypes)

print(f"\n❓ Missing Values:")
print(df.isnull().sum())

print(f"\n📈 Statistical Summary:")
df.describe()

print(f"\n🎯 Target Variable Distribution (final_result):")
print(df['final_result'].value_counts())
print(f"Pass (1): {df['final_result'].sum()} students")
print(f"Fail (0): {len(df) - df['final_result'].sum()} students")

In [ ]:
# Cell 5: EDA Visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Exploratory Data Analysis - Student Performance', fontsize=16, fontweight='bold')

# 1. Distribution of Final Results
ax1 = axes[0, 0]
result_counts = df['final_result'].value_counts()
colors = ['#2ecc71', '#e74c3c']
bars = ax1.bar(['Pass', 'Fail'], result_counts.values, color=colors, edgecolor='black')
ax1.set_title('Pass/Fail Distribution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Students')
for bar, count in zip(bars, result_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', fontweight='bold')

# 2. Study Hours vs Final Result
ax2 = axes[0, 1]
df.boxplot(column='study_hours', by='final_result', ax=ax2, patch_artist=True)
ax2.set_title('Study Hours Distribution by Result', fontsize=12, fontweight='bold')
ax2.set_xlabel('Result (0=Fail, 1=Pass)')
ax2.set_ylabel('Study Hours')
ax2.set_xticklabels(['Fail', 'Pass'])

# 3. Attendance vs Final Result
ax3 = axes[0, 2]
df.boxplot(column='attendance', by='final_result', ax=ax3, patch_artist=True)
ax3.set_title('Attendance Distribution by Result', fontsize=12, fontweight='bold')
ax3.set_xlabel('Result (0=Fail, 1=Pass)')
ax3.set_ylabel('Attendance (%)')
ax3.set_xticklabels(['Fail', 'Pass'])

# 4. Correlation Heatmap
ax4 = axes[1, 0]
numeric_cols = ['study_hours', 'attendance', 'previous_marks', 'assignments', 'internal_marks', 'final_result']
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', ax=ax4, fmt='.2f', square=True)
ax4.set_title('Feature Correlation Heatmap', fontsize=12, fontweight='bold')

# 5. Previous Marks vs Final Result
ax5 = axes[1, 1]
df.boxplot(column='previous_marks', by='final_result', ax=ax5, patch_artist=True)
ax5.set_title('Previous Marks Distribution by Result', fontsize=12, fontweight='bold')
ax5.set_xlabel('Result (0=Fail, 1=Pass)')
ax5.set_ylabel('Previous Marks')
ax5.set_xticklabels(['Fail', 'Pass'])

# 6. All Features Comparison
ax6 = axes[1, 2]
features_to_plot = ['study_hours', 'attendance', 'previous_marks', 'assignments', 'internal_marks']
pass_means = df[df['final_result'] == 1][features_to_plot].mean()
fail_means = df[df['final_result'] == 0][features_to_plot].mean()
x = np.arange(len(features_to_plot))
width = 0.35
ax6.bar(x - width/2, pass_means, width, label='Pass', color='#2ecc71')
ax6.bar(x + width/2, fail_means, width, label='Fail', color='#e74c3c')
ax6.set_title('Feature Comparison: Pass vs Fail', fontsize=12, fontweight='bold')
ax6.set_xticks(x)
ax6.set_xticklabels(features_to_plot, rotation=45)
ax6.set_ylabel('Average Value')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Data Preprocessing
print("="*50)
print("DATA PREPROCESSING")
print("="*50)

# Create a copy for preprocessing
df_processed = df.copy()

# Check for missing values
print("\n1. Checking for missing values...")
if df_processed.isnull().sum().sum() == 0:
    print("   ✅ No missing values found!")
else:
    print("   ⚠️ Missing values detected. Handling...")
    df_processed = df_processed.dropna()

# Feature Selection (according to synopsis)
feature_columns = ['study_hours', 'attendance', 'previous_marks', 'assignments', 'internal_marks']
X = df_processed[feature_columns]
y = df_processed['final_result']

print(f"\n2. Features selected: {feature_columns}")
print(f"   Feature matrix shape: {X.shape}")
print(f"   Target vector shape: {y.shape}")

# Normalize numerical values
print("\n3. Normalizing numerical features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("   ✅ Features normalized (mean=0, std=1)")

# Split dataset (80% training, 20% testing)
print("\n4. Splitting dataset...")
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"   Training data: {len(X_train)} students (80%)")
print(f"   Testing data: {len(X_test)} students (20%)")

print("\n✅ Data preprocessing complete!")

In [ ]:
# Cell 7: Model Training (All algorithms from synopsis)
print("="*50)
print("TRAINING MACHINE LEARNING MODELS")
print("="*50)

# Initialize models (all from requirements)
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Naive Bayes': GaussianNB(),
    'SVM': SVC(random_state=42, probability=True)
}

# Train and evaluate models
results = {}
trained_models = {}

for name, model in models.items():
    print(f"\n📊 Training {name}...")
    # Train model
    model.fit(X_train, y_train)
    trained_models[name] = model

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Store results
    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

    print(f"   ✅ Accuracy: {accuracy:.4f}")
    print(f"   ✅ Precision: {precision:.4f}")
    print(f"   ✅ Recall: {recall:.4f}")
    print(f"   ✅ F1-Score: {f1:.4f}")

# Create results DataFrame
results_df = pd.DataFrame(results).T
print("\n" + "="*50)
print("MODEL COMPARISON SUMMARY")
print("="*50)
print(results_df)

In [ ]:
# Cell 8: Model Performance Comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot comparison
ax1 = axes[0]
results_df.plot(kind='bar', ax=ax1, colormap='viridis', edgecolor='black')
ax1.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_xlabel('Models')
ax1.set_ylabel('Score')
ax1.set_ylim([0, 1])
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# Heatmap
ax2 = axes[1]
sns.heatmap(results_df, annot=True, cmap='YlOrRd', fmt='.4f', ax=ax2,
            cbar_kws={'label': 'Score'}, square=True)
ax2.set_title('Model Performance Heatmap', fontsize=14, fontweight='bold')
ax2.set_xlabel('Metrics')
ax2.set_ylabel('Models')

plt.tight_layout()
plt.show()

# Find best model
best_model_name = results_df['Accuracy'].idxmax()
best_accuracy = results_df.loc[best_model_name, 'Accuracy']
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"   This model will be used for predictions!")

In [ ]:
# Cell 9: Confusion Matrix
best_model = trained_models[best_model_name]
y_pred_best = best_model.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Fail (0)', 'Pass (1)'],
            yticklabels=['Fail (0)', 'Pass (1)'],
            annot_kws={'size': 16, 'weight': 'bold'})

ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)

# Add text annotations
for i in range(2):
    for j in range(2):
        ax.text(j+0.5, i+0.5, str(cm[i, j]),
                ha='center', va='center', fontsize=18, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Fail', 'Pass']))

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
print(f"\n📊 Confusion Matrix Interpretation:")
print(f"   True Positives (Correctly predicted Pass): {tp}")
print(f"   True Negatives (Correctly predicted Fail): {tn}")
print(f"   False Positives (Wrongly predicted Pass): {fp}")
print(f"   False Negatives (Wrongly predicted Fail): {fn}")

In [ ]:
# Cell 10: Feature Importance
if 'Random Forest' in trained_models:
    rf_model = trained_models['Random Forest']
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0, 1, len(feature_importance)))
    bars = ax.barh(feature_importance['Feature'], feature_importance['Importance'], color=colors, edgecolor='black')
    ax.set_title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_ylabel('Features', fontsize=12)
    ax.invert_yaxis()

    # Add value labels
    for i, (bar, importance) in enumerate(zip(bars, feature_importance['Importance'])):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{importance:.3f}', va='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("\n📊 Feature Importance Ranking:")
    print(feature_importance.to_string(index=False))
    print(f"\n💡 Top 3 Most Important Features:")
    for i, row in feature_importance.head(3).iterrows():
        print(f"   • {row['Feature']}: {row['Importance']:.3f}")
else:
    print("Feature importance available only for Random Forest model")

In [ ]:
# Cell 11: Cross-Validation
print("="*50)
print("CROSS-VALIDATION RESULTS")
print("="*50)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(best_model, X_scaled, y, cv=5, scoring='accuracy')
cv_precision = cross_val_score(best_model, X_scaled, y, cv=5, scoring='precision')
cv_recall = cross_val_score(best_model, X_scaled, y, cv=5, scoring='recall')
cv_f1 = cross_val_score(best_model, X_scaled, y, cv=5, scoring='f1')

print(f"\n📊 {best_model_name} - 5-Fold Cross-Validation:")
print(f"   Accuracy:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"   Precision: {cv_precision.mean():.4f} ± {cv_precision.std():.4f}")
print(f"   Recall:    {cv_recall.mean():.4f} ± {cv_recall.std():.4f}")
print(f"   F1-Score:  {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
cv_data = [cv_scores, cv_precision, cv_recall, cv_f1]
box_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
bp = ax.boxplot(cv_data, labels=box_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']):
    patch.set_facecolor(color)
ax.set_title(f'Cross-Validation Distribution - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Prediction Function
def predict_student_performance(model, scaler, student_data):
    """
    Predict performance for a single student
    Based on synopsis requirements
    """
    # Extract features in correct order
    features = [
        student_data['study_hours'],
        student_data['attendance'],
        student_data['previous_marks'],
        student_data['assignments'],
        student_data['internal_marks']
    ]

    # Scale features
    features_scaled = scaler.transform([features])

    # Predict
    prediction = model.predict(features_scaled)[0]

    # Get probability if available
    if hasattr(model, 'predict_proba'):
        probability = model.predict_proba(features_scaled)[0]
        confidence = probability[1] if prediction == 1 else probability[0]
    else:
        confidence = 0.5

    result = 'Pass' if prediction == 1 else 'Fail'

    return result, confidence

print("="*50)
print("SINGLE STUDENT PREDICTION")
print("="*50)

# Test with different student profiles
test_students = [
    {'study_hours': 8, 'attendance': 95, 'previous_marks': 90, 'assignments': 92, 'internal_marks': 88},
    {'study_hours': 2, 'attendance': 45, 'previous_marks': 35, 'assignments': 40, 'internal_marks': 38},
    {'study_hours': 5, 'attendance': 70, 'previous_marks': 65, 'assignments': 68, 'internal_marks': 66},
    {'study_hours': 7, 'attendance': 85, 'previous_marks': 78, 'assignments': 80, 'internal_marks': 79},
]

for i, student in enumerate(test_students, 1):
    result, confidence = predict_student_performance(best_model, scaler, student)
    print(f"\n📝 Student {i}:")
    print(f"   Study Hours: {student['study_hours']} hrs/day")
    print(f"   Attendance: {student['attendance']}%")
    print(f"   Previous Marks: {student['previous_marks']}")
    print(f"   Assignments: {student['assignments']}")
    print(f"   Internal Marks: {student['internal_marks']}")
    print(f"   → Prediction: {result}")
    print(f"   → Confidence: {confidence:.2%}")

In [ ]:
# Cell 13: Identify At-Risk Students
print("="*50)
print("AT-RISK STUDENT IDENTIFICATION")
print("="*50)

# Use best model to predict on all students
all_predictions = best_model.predict(X_scaled)
if hasattr(best_model, 'predict_proba'):
    all_probs = best_model.predict_proba(X_scaled)[:, 1]
else:
    all_probs = [0.5] * len(df)

# Add predictions to dataframe
df['Prediction'] = ['Pass' if p == 1 else 'Fail' for p in all_predictions]
df['Confidence'] = all_probs
df['Risk_Level'] = ['High Risk' if (p == 0 and prob > 0.7) else
                     'Medium Risk' if (p == 0 and prob > 0.5) else
                     'Low Risk' for p, prob in zip(all_predictions, all_probs)]

# Identify at-risk students (predicted as fail)
at_risk_students = df[df['Prediction'] == 'Fail'].copy()
if len(at_risk_students) > 0:
    at_risk_students = at_risk_students.sort_values('Confidence', ascending=False)
else:
    at_risk_students = pd.DataFrame()

print(f"\n📊 Summary:")
print(f"   Total Students: {len(df)}")
print(f"   ✅ Predicted to Pass: {len(df[df['Prediction'] == 'Pass'])}")
print(f"   ⚠️  Predicted to Fail (At-Risk): {len(at_risk_students)}")

if len(at_risk_students) > 0:
    print(f"\n🔴 AT-RISK STUDENTS (Need Immediate Attention):")
    display_cols = ['student_id', 'study_hours', 'attendance', 'previous_marks',
                    'assignments', 'internal_marks', 'Confidence', 'Risk_Level']
    print(at_risk_students[display_cols].to_string(index=False))

    # Visualize at-risk students
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Comparison of features
    ax1 = axes[0]
    features_to_plot = ['study_hours', 'attendance', 'previous_marks', 'assignments', 'internal_marks']
    means_df = pd.DataFrame({
        'At-Risk Students': at_risk_students[features_to_plot].mean(),
        'All Students': df[features_to_plot].mean()
    })
    means_df.plot(kind='bar', ax=ax1, color=['#e74c3c', '#3498db'], edgecolor='black')
    ax1.set_title('Feature Comparison: At-Risk vs All Students', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Average Value')
    ax1.set_xlabel('Features')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)

    # Risk distribution
    ax2 = axes[1]
    risk_counts = df['Risk_Level'].value_counts()
    risk_colors = {'High Risk': '#e74c3c', 'Medium Risk': '#f39c12', 'Low Risk': '#2ecc71'}
    bars = ax2.bar(risk_counts.index, risk_counts.values, color=[risk_colors[r] for r in risk_counts.index], edgecolor='black')
    ax2.set_title('Student Risk Level Distribution', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Number of Students')
    ax2.set_xlabel('Risk Level')
    for bar, count in zip(bars, risk_counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(count), ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("\n💡 RECOMMENDATIONS FOR AT-RISK STUDENTS:")
    print("   1. Provide additional tutoring sessions")
    print("   2. Increase study hours monitoring")
    print("   3. Improve attendance tracking")
    print("   4. Offer remedial classes")
    print("   5. Regular parent-teacher meetings")
else:
    print("\n✅ No at-risk students found! All students are predicted to pass.")

In [ ]:
# Cell 14: Complete Performance Dashboard
fig = plt.figure(figsize=(20, 14))
fig.suptitle('STUDENT PERFORMANCE PREDICTION SYSTEM - COMPLETE DASHBOARD',
             fontsize=18, fontweight='bold', y=0.98)

# 1. Pass/Fail Distribution (Top Left)
ax1 = plt.subplot(3, 3, 1)
pass_fail = df['final_result'].value_counts()
colors_pie = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax1.pie(pass_fail, labels=['Pass', 'Fail'],
                                    autopct='%1.1f%%', colors=colors_pie,
                                    startangle=90, explode=(0.05, 0.05))
ax1.set_title('Actual Performance Distribution', fontsize=12, fontweight='bold')

# 2. Model Performance (Top Middle)
ax2 = plt.subplot(3, 3, 2)
results_df.plot(kind='bar', ax=ax2, colormap='Set2', edgecolor='black', legend=True)
ax2.set_title('Model Performance Comparison', fontsize=12, fontweight='bold')
ax2.set_ylabel('Score')
ax2.set_ylim([0, 1])
ax2.set_xlabel('')
ax2.legend(loc='lower right', fontsize=8)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

# 3. At-Risk Summary (Top Right)
ax3 = plt.subplot(3, 3, 3)
risk_summary = [len(df) - len(at_risk_students), len(at_risk_students)] if len(at_risk_students) > 0 else [len(df), 0]
risk_labels = ['Safe', 'At-Risk']
risk_colors = ['#3498db', '#e74c3c']
bars = ax3.bar(risk_labels, risk_summary, color=risk_colors, edgecolor='black', linewidth=2)
ax3.set_title('Student Risk Assessment', fontsize=12, fontweight='bold')
ax3.set_ylabel('Number of Students')
for bar, count in zip(bars, risk_summary):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', fontweight='bold', fontsize=12)

# 4. Study Hours Distribution (Middle Left)
ax4 = plt.subplot(3, 3, 4)
df.boxplot(column='study_hours', by='Prediction', ax=ax4, patch_artist=True)
ax4.set_title('Study Hours: Pass vs Fail Prediction', fontsize=12, fontweight='bold')
ax4.set_xlabel('Prediction')
ax4.set_ylabel('Study Hours')
ax4.set_xticklabels(['Fail', 'Pass'])

# 5. Attendance Distribution (Middle Center)
ax5 = plt.subplot(3, 3, 5)
df.boxplot(column='attendance', by='Prediction', ax=ax5, patch_artist=True)
ax5.set_title('Attendance: Pass vs Fail Prediction', fontsize=12, fontweight='bold')
ax5.set_xlabel('Prediction')
ax5.set_ylabel('Attendance (%)')
ax5.set_xticklabels(['Fail', 'Pass'])

# 6. Confusion Matrix (Middle Right)
ax6 = plt.subplot(3, 3, 6)
cm_best = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=ax6,
            xticklabels=['Fail', 'Pass'],
            yticklabels=['Fail', 'Pass'],
            annot_kws={'size': 14})
ax6.set_title(f'Confusion Matrix - {best_model_name}', fontsize=12, fontweight='bold')
ax6.set_xlabel('Predicted')
ax6.set_ylabel('Actual')

# 7. Feature Importance (Bottom Left)
ax7 = plt.subplot(3, 3, 7)
if 'Random Forest' in trained_models:
    feature_importance_sorted = feature_importance.sort_values('Importance', ascending=True)
    ax7.barh(feature_importance_sorted['Feature'], feature_importance_sorted['Importance'],
             color='#3498db', edgecolor='black')
    ax7.set_title('Feature Importance Ranking', fontsize=12, fontweight='bold')
    ax7.set_xlabel('Importance Score')
    ax7.set_ylabel('Features')

# 8. Prediction Accuracy Gauge (Bottom Center)
ax8 = plt.subplot(3, 3, 8)
accuracy_pct = best_accuracy * 100
colors_gauge = ['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71']
gauge_data = [25, 25, 25, 25]
gauge_labels = [f'0-25%', f'25-50%', f'50-75%', f'75-100%']
wedges, texts, autotexts = ax8.pie(gauge_data, labels=gauge_labels, colors=colors_gauge,
                                    startangle=90, autopct='')
# Add center text
center_circle = plt.Circle((0, 0), 0.70, fc='white')
fig.gca().add_artist(center_circle)
ax8.text(0, 0, f'{accuracy_pct:.1f}%', ha='center', va='center', fontsize=20, fontweight='bold')
ax8.set_title(f'Model Accuracy: {best_model_name}', fontsize=12, fontweight='bold')

# 9. Cross-Validation Results (Bottom Right)
ax9 = plt.subplot(3, 3, 9)
cv_means = [cv_scores.mean(), cv_precision.mean(), cv_recall.mean(), cv_f1.mean()]
cv_stds = [cv_scores.std(), cv_precision.std(), cv_recall.std(), cv_f1.std()]
x_pos = np.arange(len(box_labels))
ax9.bar(x_pos, cv_means, yerr=cv_stds, capsize=5, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'],
        edgecolor='black', alpha=0.7)
ax9.set_title('Cross-Validation Results (5-Fold)', fontsize=12, fontweight='bold')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(box_labels)
ax9.set_ylabel('Score')
ax9.set_ylim([0, 1])
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("PROJECT EXECUTION COMPLETE!")
print("="*50)

In [ ]:
# Cell 17: Simple UI - Alternative Version (No Errors, Very Lightweight)
import ipywidgets as widgets
from IPython.display import display, clear_output

print("="*60)
print("🎓 SIMPLE STUDENT PERFORMANCE PREDICTOR")
print("="*60)

# Create simple widgets
study = widgets.FloatSlider(value=5, min=0, max=10, step=0.5, description='Study Hours:')
attend = widgets.IntSlider(value=75, min=0, max=100, step=1, description='Attendance:')
prev = widgets.IntSlider(value=65, min=0, max=100, step=1, description='Previous Marks:')
assign = widgets.IntSlider(value=70, min=0, max=100, step=1, description='Assignments:')
internal = widgets.IntSlider(value=68, min=0, max=100, step=1, description='Internal Marks:')

output = widgets.Output()

def predict_simple(study, attend, prev, assign, internal):
    with output:
        clear_output(wait=True)

        # Simple prediction logic
        score = (study/10 * 0.3 + attend/100 * 0.3 + prev/100 * 0.2 +
                assign/100 * 0.1 + internal/100 * 0.1)

        if score >= 0.5:
            result = "✅ PASS"
            color = "green"
            risk = "Low Risk"
            recommendation = "Keep up the good work!"
        else:
            result = "❌ FAIL"
            color = "red"
            risk = "High Risk"
            recommendation = "Need improvement! Increase study hours."

        print(f"\n{'='*50}")
        print(f"📊 PREDICTION RESULT")
        print(f"{'='*50}")
        print(f"Result: {result}")
        print(f"Confidence: {score:.1%}")
        print(f"Risk Level: {risk}")
        print(f"\n📈 Performance Metrics:")
        print(f"  • Study Efficiency: {study/10*100:.0f}%")
        print(f"  • Attendance Score: {attend}%")
        print(f"  • Academic Score: {(prev+assign+internal)/3:.0f}")
        print(f"\n💡 Recommendation: {recommendation}")
        print(f"{'='*50}")

# Create UI
ui = widgets.VBox([
    widgets.HTML("<h2>🎓 Student Performance Predictor</h2>"),
    study, attend, prev, assign, internal,
    widgets.Button(description='Predict', button_style='primary')
])

# Connect button
btn = ui.children[-1]
def on_click(b):
    predict_simple(study.value, attend.value, prev.value, assign.value, internal.value)
btn.on_click(on_click)

display(ui, output)